# Clase 3 - Analisis de Datos

## Libraries

---

Para leer distintas bases de datos, usaremos dos nuevas bilbiotecas:

- openpyxl
- lxml

(primero las instalamos usando el comando pip directo el la shell)

In [1]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Ahora importamos las librerias

In [3]:
import pandas as pd
import numpy as np
import re
import openpyxl

## Funciones

---

Dado que la base de datos a utilizar (archivo de excel del INEGI) no tiene la estructura adecuada, usaremos funciones para limpiar y transformarla.

### Transformando y limpiando los datos



In [75]:
def fix_periodo_ano_trimestre(df_in, col_tiempo):
    df = df_in.copy()
    p = df[col_tiempo].astype(str).str.strip()    # We remove the leading and trailing whitespaces in the values of the Series associated to the column col_tiempo
    is_year = p.str.fullmatch(r"\d{4}")
    rom_map = {"I": 1, "II": 2, "III": 3, "IIII": 4}    # We will use this dictionary to map roman numeral to decimal numbers
    is_q = p.isin(rom_map.keys())       # The quarters are the previous diccionary's keys
    df["_anio"] = np.where(is_year, p, np.nan)      # the np.where() returns elements depending on the condition (the TRUE associated element is p, and the false associated element np.nan)
                                                    # this essentially replaces the values of the column "_anio" with the 4 digit year if it is indeed 4 digits, and NaN if otherwise
    df["_anio"] = df["_anio"].ffill()    # The ffill mehod fills NA/NaN values by propagating the last valid observation to next valid.
    df["_q"] = np.where(is_q, p.map(rom_map), np.nan)    # This will transform the roman numerals to decimal numbers and leave the 4 digit years intact
    df.loc[is_q, col_tiempo] = df.loc[is_q, "_anio"].astype(str) + "_Q" + df.loc[is_q, "_q"].astype(int).astype(str) # Very interesting line, see the detailed comments below

    df = df[is_q].copy()
    df = df[[x for x in df.columns.astype(str).tolist() if x not in ["_q", "_anio"]]].reset_index(drop = True)
    # df = df.drop(columns = ["_q", "_anio"]).reset_index(drop = True)    # This does the same but using the .drop() method
    return df

### Explanation of the previous function

DataFrames are mutable objects. As such, it is important to be cautious when doing transformations on them, as the change is visible through all its names. To prevent that, in this funcion we used `df_in.copy()`; we use a copy of the orignal dataframe, essentilly creating a second identical object that is stored in memory as one different than the original. 

In line *4* we use the pandas method str.fullmatch to filter the values of the striped "col_tiempo" column. We verify that it matched the pattern we specified using regular expressions of it having exactly 4 digits.

In line *7* we use create a temporal column "_anio" (since is temporal, we use a lowerbar at the beggining); we use the .where method (similar to an if that acts on the whole Series) to leave all the values except the ones that are 4 digits, as NAs. Consequelntly, in line *9* we use the ffill (forward fill) method to fill the NaN values by propagating the last valid value (the 4 digit years) until the next valid value is met.

In line *10* we create a new column "_q" (q from quarters) where we place the parsed decimals, and NAs when no roman numeral were found.

Line *11* is the most interesting and complex one. Here is where we use the data that we placed in the two previous newly created columns to change the values in the ocol_tiempo column that were originally roman numerals. What we do is that we concatenate the year, the string "_Q" and the parsed decimal. That way the values of the column will be of the form [2000, 2000_Q1, 2000_Q2, ...].

Finaly, in the last 3 lines of the function, we copy the filtered DataFrame. Since only the rows that had roman numeral at the beggining have meaninfull data, we filter using is_q, essentially copyting only the rows of the dataframe that have meaningfull data. Note that now the values of the column col_tiempo are of the previously exposed form, and the DataFrame still has the extra temporal columns "_anio" and "_q". It is here that we filter again, with the intention of 'erasing' the aforementioned temporal columns. We then reset the index. Also note that now that we have finished (momentarily) transforming the DataFrame (the copy we made at the beggining), we can go back now to reasing the name of the original DataFrame to this newly tranformed one. It returns said DataFrame

## Input

---

To read the .xls file that contains the info, we cannot use pd.read_csv. In this case it is read as:

In [5]:
df_inegi_desocupacion = pd.read_html("../INPUTS/Tabulado.xls")    # This is reading a list of arrays
display(df_inegi_desocupacion)

[                                             Instituto Nacional de Estadística y Geografía (INEGI)  \
                                                                                      Desocupación.   
    Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa   
                                                                     (Porcentaje respecto a la PEA)   
                                                                                            Periodo   
 0                                                2022                                                
 1                                                   I                                                
 2                                                  II                                                
 3                                                 III                                                
 4                                                  IV                   

In [6]:
df_inegi_desocupacion[0]

Instituto Nacional de Estadística y Geografía (INEGI)  \
                                                                                     Desocupación.   
   Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa   
                                                                    (Porcentaje respecto a la PEA)   
                                                                                           Periodo   
0                                                2022                                                
1                                                   I                                                
2                                                  II                                                
3                                                 III                                                
4                                                  IV                                                
5                                                2023                                                
6                                                   I                                                
7                                                  II                                                
8                                                 III                                                
9                                                  IV                                                
10                                               2024                                                
11                                                  I                                                
12                                                 II                                                
13                                                III                                                
14                                                 IV                                                
15                                               2025                                                
16                                                  I                                                
17                                                 II                                                
18                                                III                                                
19  Fuente: INEGI. Series calculadas por métodos e...                                                
20                                                NaN                                                

                                                                 \
                                                                  
                                                                  
                                                                  
   Aguascalientes Baja California Baja California Sur  Campeche   
0     2022.000000     2022.000000         2022.000000  2022.000   
1        3.953469        2.134665            2.992033     3.068   
2        3.798665        2.427478            2.785745     2.953   
3        3.697704        2.304535            2.555263     2.298   
4        3.652696        2.835179            3.071859     1.976   
5     2023.000000     2023.000000         2023.000000  2023.000   
6        3.460535        1.936176            2.848725     1.762   
7        3.183940        2.215882            2.740515     1.759   
8        2.890444        2.188865            2.849253     1.819   
9        3.417412        2.327144            2.449391     1.715   
10    2024.000000     2024.000000         2024.000000  2024.000   
11       3.489829        2.411825            2.282422     1.563   
12       3.298298        2.635777            2.447040     1.998   
13       3.839646        2.451764            2.254498     1.951   
14       2.227024        2.530388            2.050320     1.897   
15    2025.000000     2025.000000         2025.00000

Observe that this won't work for us. When reading the file as html, we essentially obtain a list of arrays; and we need something that can be transformed into a DataFrame. Note that the original file one has one array of data, and hence the list of arrays will be a list whose only element is an array. If we speficy that we only want the first elemnt of the list, we will succesfully obtain the array we are insterested in.

We do this next:

In [7]:
df_inegi_desocupacion_v2 = df_inegi_desocupacion[0].copy()
display(df_inegi_desocupacion_v2)   # We continue making copies to not mess up the data we know has not incurred in processing errors of our making.

Instituto Nacional de Estadística y Geografía (INEGI)  \
                                                                                     Desocupación.   
   Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa   
                                                                    (Porcentaje respecto a la PEA)   
                                                                                           Periodo   
0                                                2022                                                
1                                                   I                                                
2                                                  II                                                
3                                                 III                                                
4                                                  IV                                                
5                                                2023                                                
6                                                   I                                                
7                                                  II                                                
8                                                 III                                                
9                                                  IV                                                
10                                               2024                                                
11                                                  I                                                
12                                                 II                                                
13                                                III                                                
14                                                 IV                                                
15                                               2025                                                
16                                                  I                                                
17                                                 II                                                
18                                                III                                                
19  Fuente: INEGI. Series calculadas por métodos e...                                                
20                                                NaN                                                

                                                                 \
                                                                  
                                                                  
                                                                  
   Aguascalientes Baja California Baja California Sur  Campeche   
0     2022.000000     2022.000000         2022.000000  2022.000   
1        3.953469        2.134665            2.992033     3.068   
2        3.798665        2.427478            2.785745     2.953   
3        3.697704        2.304535            2.555263     2.298   
4        3.652696        2.835179            3.071859     1.976   
5     2023.000000     2023.000000         2023.000000  2023.000   
6        3.460535        1.936176            2.848725     1.762   
7        3.183940        2.215882            2.740515     1.759   
8        2.890444        2.188865            2.849253     1.819   
9        3.417412        2.327144            2.449391     1.715   
10    2024.000000     2024.000000         2024.000000  2024.000   
11       3.489829        2.411825            2.282422     1.563   
12       3.298298        2.635777            2.447040     1.998   
13       3.839646        2.451764            2.254498     1.951   
14       2.227024        2.530388            2.050320     1.897   
15    2025.000000     2025.000000         2025.00000

Thus this should return an error

In [8]:
df_inegi_desocupacion[1] # Observe how this returns an error message

IndexError: list index out of range

If it were a regular excel file, we would read is with:

In [ ]:
df_alt = pd.read_excel("../INPUTS/Tabulado_v2.xlsx")
display(df_alt.head(10))

,Instituto Nacional de Estadística y Geografía (INEGI),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Desocupación.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Series desestacionalizadas de la tasa de desoc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(Porcentaje respecto a la PEA),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Periodo,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,Ciudad de México,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
6,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,I,3.953469,2.134665,2.992033,3.068,4.866799,2.872954,2.522971,2.479422,6.06356,...,3.136971,3.405845,2.939622,3.088067,5.042098,3.880562,4.292356,2.924682,2.13913,2.627598
8,II,3.798665,2.427478,2.785745,2.953,4.131998,2.661397,2.709404,2.380596,5.266869,...,2.900986,3.248934,3.01812,3.119569,5.600978,3.416105,4.032327,2.954154,1.785905,3.140979
9,III,3.697704,2.304535,2.555263,2.298,4.051126,2.462578,2.189561,2.312669,5.057697,...,2.807988,2.975509,2.908285,3.398284,4.985484,3.468522,3.509932,2.856816,1.894844,2.823368


In [87]:
df_pob = pd.read_csv("../INPUTS/INE_ENTIDAD_2020.csv", encoding="latin-1")
df_pob

,ENT,NOM_ENT,POBTOT,POBFEM,POBMAS,P_0A2,P_0A2_F,P_0A2_M,P_0A17,P_3YMAS,...,VPH_TELEF,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCIN,VPH_SINTIC
0,1,Aguascalientes,1425607,728924,696683,71864,35604,36260,463335,1352235,...,147818,359895,236003,174089,98724,70126,6021,15323,128996,1711
1,2,Baja California,3769020,1868431,1900589,149957,74010,75947,1061893,3610844,...,576806,1080169,800189,618175,384011,216865,41223,38772,293529,9582
2,3,Baja California Sur,798447,392568,405879,35963,17821,18142,236629,758642,...,91750,226517,148723,136538,67961,36197,14508,8675,77223,2608
3,4,Campeche,928363,471424,456939,45541,22506,23035,286880,878528,...,68843,218322,114020,151613,38508,17976,23627,36397,130361,12028
4,5,Coahuila de Zaragoza,3146771,1583102,1563669,160368,79095,81273,982841,2980244,...,372132,824291,519599,443659,195883,124077,17020,46420,332298,5754
5,6,Colima,731391,370769,360622,29614,14483,15131,209402,699821,...,80251,206736,132395,114164,43881,22695,9173,12085,82366,2698
6,7,Chiapas,5543828,2837881,2705947,334055,165892,168163,2093426,5181929,...,159652,944695,292189,433400,61298,32460,214333,379915,993929,151655
7,8,Chihuahua,3741869,1888047,1853822,163950,80466,83484,1137537,3570280,...,460540,1051045,650546,481871,282049,179964,42113,64419,432032,20158
8,9,Ciudad de México,9209944,4805017,4404927,267151,131720,135431,2033581,8933167,...,1898265,2536523,2084156,1290811,957162,568827,46172,77272,561128,10528
9,10,Durango,1832650,927784,904866,95793,47204,48589,612346,1733682,...,137873,434450,215108,222785,73099,54664,22071,41801,243750,12446


## Evaluation

---


Note that the structure of the DataFrame is not ideal. To see why just observe the result of using the .columns method

In [ ]:
df_inegi_desocupacion_v2.columns

MultiIndex([('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            ('Instituto Nacional de Estadística y Geografía (INEGI)', ...),
            (                                  'Unnamed: 10_level_0', ...),
            (                                  'Unnamed: 11_level_0', ...),
            (                                  'Unnamed: 12_level_0', ...),
            

In [ ]:
df_inegi_desocupacion_v2.columns[0]

('Instituto Nacional de Estadística y Geografía (INEGI)',
 'Desocupación.',
 'Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa',
 '(Porcentaje respecto a la PEA)',
 'Periodo')

In [ ]:
df_inegi_desocupacion_v2.columns[1]

('Instituto Nacional de Estadística y Geografía (INEGI)',
 'Desocupación.',
 'Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa',
 '(Porcentaje respecto a la PEA)',
 'Aguascalientes')

In [ ]:
df_inegi_desocupacion_v2.columns[10]

('Unnamed: 10_level_0',
 'Unnamed: 10_level_1',
 'Unnamed: 10_level_2',
 'Unnamed: 10_level_3',
 'Durango')

The last element being the 33th:

In [ ]:
df_inegi_desocupacion_v2.columns[32]

('Unnamed: 32_level_0',
 'Unnamed: 32_level_1',
 'Unnamed: 32_level_2',
 'Unnamed: 32_level_3',
 'Zacatecas')

### How is the DataFrame structured?

To truly understand what will be done here onwards, it is imperative to understand what is the structure of the DataFrame so far. Here we give a detailed description of sich structure:

#### Understanding the first 5 rows in the xls file

If one opens the [file](../INPUTS/Tabulado.xls), it is easy to see that the first 5 rows have no meaningfull data. They contain the title of the data and some information about its originis. If we pay close attention we'll see that there are merged cells all over the place. The rows 1 and 2 have merged cells from the column A until the column J; and the next four rows (3 to 6) are each merged from the column A until J (see the next screenshot for reference).

![imagen](../ATTACHMENTS/imagen_01_clase_03.jpg)

Now that we've noticed this, it is easy to see the structure of the generated DataFrame.

### Understangin the columns of the DataFrame - A multiindex issue

Usinf the .info method we can get a summary of the DataFrame

In [ ]:
df_inegi_desocupacion_v2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 33 columns):
 #   Column                                                                                                                                                                                                                         Non-Null Count  Dtype  
---  ------                                                                                                                                                                                                                         --------------  -----  
 0   (Instituto Nacional de Estadística y Geografía (INEGI), Desocupación., Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa, (Porcentaje respecto a la PEA), Periodo)               20 non-null     object 
 1   (Instituto Nacional de Estadística y Geografía (INEGI), Desocupación., Series desestacionalizadas de la tasa de desocupación total trime

Reading the output of the .info method, we can see there are 33 columns: one for the "Periodo" column, and 32 for the 32 Mexican states. 

But if we recall from before, when we called the .columns method we got a list of tuples (33 tuples to be exact). Each of those tuples had 6 entries, the last one being either "Periodo" or one of Mexico's states. Why is that so? Why are the columns defined in this way?

Understanding this is simple once we see the previous image. What pandas is parsing is that the actual data doesn't start until one row below the row with the names of the states. So all the data in the columns above consitute a single column. For example, for the column[0] we have that the first 5 rows are 

- 'Instituto Nacional de Estadística y Geografía (INEGI)',
- 'Desocupación.',
- 'Series desestacionalizadas de la tasa de desocupación total trimestral según entidad federativa',
- '(Porcentaje respecto a la PEA)'
- 'Periodo'

For the second columns (column[1]) we have the same, but instead of 'Periodo' we have 'Aguascalientes'.

This same pattern continues until reaching the Kth column (which is represented by column[10]). Here the horizontal merge of the first 5 cells is no more, so we have 5 empty cells (each representing a row ), and then the name of a state. Since no data is stored in those empty cells, pandas parses a tuple with the following strings:

- 'Unnamed: 10_level_0',
- 'Unnamed: 10_level_1',
- 'Unnamed: 10_level_2',
- 'Unnamed: 10_level_3',
- 'Zacatecas"

This repeats for all the following columns.

This can be formally understood as the DataFrame having a column multi-index (and vecto-like index of size n>1).

## Transformation

---

We need to fix the issue with the column index. Its values should not be defined as a tuple with 5 entries; their actual value should be the last entry of said tuples. We need then to transform the index column. We do this next:

In [10]:
hdr = df_inegi_desocupacion_v2.columns.get_level_values(-1).astype(str).str.strip().str.replace(r"\s+", " ", regex = True).tolist()
hdr

['Periodo',
 'Aguascalientes',
 'Baja California',
 'Baja California Sur',
 'Campeche',
 'Coahuila de Zaragoza',
 'Colima',
 'Chiapas',
 'Chihuahua',
 'Ciudad de México',
 'Durango',
 'Guanajuato',
 'Guerrero',
 'Hidalgo',
 'Jalisco',
 'México',
 'Michoacán de Ocampo',
 'Morelos',
 'Nayarit',
 'Nuevo León',
 'Oaxaca',
 'Puebla',
 'Querétaro',
 'Quintana Roo',
 'San Luis Potosí',
 'Sinaloa',
 'Sonora',
 'Tabasco',
 'Tamaulipas',
 'Tlaxcala',
 'Veracruz de Ignacio de la Llave',
 'Yucatán',
 'Zacatecas']

This last line of code does quite a lot of things; here we stop to explain it:

First, by the previous section we know that the column index is actually a multindex. The goal is to turn it into a regular column index, and for that we first need to define a list that contains all the new column names. To do so, we first use the .get_level_values method on the df_inegi_desocipacion_v2.columns object. This method is useful to get an individual level of values from a multi-index; we specify -1 to get the last item (remember that we can use negative indeces starting from -1 that swipe the list-like object backwards starting from the end). We then cast said value to a string data type, remove trailing and leading whitespaces with the .strip method, and use regular expressions to remove double spaces (Here we use the regular expression r"\s+", the s stand for space, and the + specifies that more than one continuos space meets the criteria). We then cast the resulting object into a list.

This way we'll have a list with all the values we actually want for the column index.

We then create another copy of the DataFrame and change the column index to have the values of the list hdr.

In [19]:
df_inegi_desocupacion_v3 = df_inegi_desocupacion_v2
df_inegi_desocupacion_v3.columns = hdr
df_inegi_desocupacion_v3.columns = pd.Index(df_inegi_desocupacion_v3.columns).astype(str).str.replace("\n", " ").str.strip().str.replace(r"\s+", " ", regex = True)   # Why this line?
df_inegi_desocupacion_v3 = df_inegi_desocupacion_v3.loc[:,~df_inegi_desocupacion_v3.columns.str.startswith("Unnamed", na = False)].copy()   # Validamos para que no traiga ninguna columna que empieze con
#                                                                                                                                            # "Unamed" (redundant)

display(df_inegi_desocupacion_v3)

,Periodo,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,Ciudad de México,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
0,2022,2022.000000,2022.000000,2022.000000,2022.000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,...,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000
1,I,3.953469,2.134665,2.992033,3.068,4.866799,2.872954,2.522971,2.479422,6.063560,...,3.136971,3.405845,2.939622,3.088067,5.042098,3.880562,4.292356,2.924682,2.139130,2.627598
2,II,3.798665,2.427478,2.785745,2.953,4.131998,2.661397,2.709404,2.380596,5.266869,...,2.900986,3.248934,3.018120,3.119569,5.600978,3.416105,4.032327,2.954154,1.785905,3.140979
3,III,3.697704,2.304535,2.555263,2.298,4.051126,2.462578,2.189561,2.312669,5.057697,...,2.807988,2.975509,2.908285,3.398284,4.985484,3.468522,3.509932,2.856816,1.894844,2.823368
4,IV,3.652696,2.835179,3.071859,1.976,3.575970,2.327980,2.296400,2.289159,4.654509,...,1.933725,2.733978,2.827257,2.604291,4.274609,3.211365,3.512586,2.238432,2.027974,2.582576
5,2023,2023.000000,2023.000000,2023.000000,2023.000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,...,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000
6,I,3.460535,1.936176,2.848725,1.762,3.554704,2.217715,2.485124,2.720658,3.900855,...,2.679430,2.339895,2.266381,3.122648,4.106569,2.853784,3.192391,2.756010,1.952460,3.211990
7,II,3.183940,2.215882,2.740515,1.759,3.814756,2.287556,1.951570,2.395119,4.255903,...,2.413697,2.766792,2.277819,2.661481,3.938371,3.117355,3.580712,2.130368,1.687909,2.786777
8,III,2.890444,2.188865,2.849253,1.819,3.674084,2.410967,2.426217,2.619587,3.930840,...,2.623781,2.724866,2.618993,2.590500,3.722493,3.170172,3.374651,1.848899,1.903655,2.470896
9,IV,3.417412,2.327144,2.449391,1.715,4.513674,2.283144,1.670047,2.194779,3.978990,...,2.565495,3.088624,2.314196,2.881328,3.949906,3.356906,3.181382,1.817428,1.812354,3.017325


Now the DataFrame has a much more clerer and clean structure. Still we can see that the last two rows have no data, and hence must be removed.

Using the .info method we can also get a clear picture of the fields and rows

In [20]:
df_inegi_desocupacion_v3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Periodo                          20 non-null     object 
 1   Aguascalientes                   19 non-null     float64
 2   Baja California                  19 non-null     float64
 3   Baja California Sur              19 non-null     float64
 4   Campeche                         19 non-null     float64
 5   Coahuila de Zaragoza             19 non-null     float64
 6   Colima                           19 non-null     float64
 7   Chiapas                          19 non-null     float64
 8   Chihuahua                        19 non-null     float64
 9   Ciudad de México                 19 non-null     float64
 10  Durango                          19 non-null     float64
 11  Guanajuato                       19 non-null     float64
 12  Guerrero                

By analysing the information summary we can see that there are null values in every column (for there are 21 entries). That is, we still need to perform a few transformations before we can start analysing the data.

To avoid hard coding this transformation we can do:

In [33]:
df_inegi_desocupacion_v4 = df_inegi_desocupacion_v3.copy()
df_inegi_desocupacion_v4 = df_inegi_desocupacion_v4[df_inegi_desocupacion_v4[df_inegi_desocupacion_v3.columns[1]].isna() == False]
display(df_inegi_desocupacion_v4)

,Periodo,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,Ciudad de México,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
0,2022,2022.000000,2022.000000,2022.000000,2022.000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,...,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000,2022.000000
1,I,3.953469,2.134665,2.992033,3.068,4.866799,2.872954,2.522971,2.479422,6.063560,...,3.136971,3.405845,2.939622,3.088067,5.042098,3.880562,4.292356,2.924682,2.139130,2.627598
2,II,3.798665,2.427478,2.785745,2.953,4.131998,2.661397,2.709404,2.380596,5.266869,...,2.900986,3.248934,3.018120,3.119569,5.600978,3.416105,4.032327,2.954154,1.785905,3.140979
3,III,3.697704,2.304535,2.555263,2.298,4.051126,2.462578,2.189561,2.312669,5.057697,...,2.807988,2.975509,2.908285,3.398284,4.985484,3.468522,3.509932,2.856816,1.894844,2.823368
4,IV,3.652696,2.835179,3.071859,1.976,3.575970,2.327980,2.296400,2.289159,4.654509,...,1.933725,2.733978,2.827257,2.604291,4.274609,3.211365,3.512586,2.238432,2.027974,2.582576
5,2023,2023.000000,2023.000000,2023.000000,2023.000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,...,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000,2023.000000
6,I,3.460535,1.936176,2.848725,1.762,3.554704,2.217715,2.485124,2.720658,3.900855,...,2.679430,2.339895,2.266381,3.122648,4.106569,2.853784,3.192391,2.756010,1.952460,3.211990
7,II,3.183940,2.215882,2.740515,1.759,3.814756,2.287556,1.951570,2.395119,4.255903,...,2.413697,2.766792,2.277819,2.661481,3.938371,3.117355,3.580712,2.130368,1.687909,2.786777
8,III,2.890444,2.188865,2.849253,1.819,3.674084,2.410967,2.426217,2.619587,3.930840,...,2.623781,2.724866,2.618993,2.590500,3.722493,3.170172,3.374651,1.848899,1.903655,2.470896
9,IV,3.417412,2.327144,2.449391,1.715,4.513674,2.283144,1.670047,2.194779,3.978990,...,2.565495,3.088624,2.314196,2.881328,3.949906,3.356906,3.181382,1.817428,1.812354,3.017325


The 2nd line of the previous block of code reassings the copy v4 of the DataFrame to a filtered version of itself. The filter of said filtered version only includes the names of the columns that have no NAs (without counting the first column). 

- df_inegi_desocupacion_v3.columns[1]: This selects the second column of the DataFrame v3
- df_inegi_desocupacion_v4[df_inegi_desocupacion_v3.columns[1]].isna() == False: This yields a Bool Series that has True values if the associated value in the 2nd column of df_v3 is non NaN, and False if it is Nan.
-  df_inegi_desocupacion_v4[df_inegi_desocupacion_v4[df_inegi_desocupacion_v3.columns[1]].isna() == False] : This functions as a .loc. It brings all the rows of the series that are TRUE (i.e. leaving out the last 2).

> PENDIENTE - Why is it the same as .loc? Isnt this structure for column retireval?

### Application of the function fix to homogenize the year - quarter data

Now that the DataFrame has the right size, we can use the function we difined in the beggining to add the year and parse the roman numerals.

In [76]:
df_inegi_desocupacion_v5 = df_inegi_desocupacion_v4.copy()
df_inegi_desocupacion_v5 = fix_periodo_ano_trimestre(df_inegi_desocupacion_v5, "Periodo")
display(df_inegi_desocupacion_v5)

,Periodo,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,Ciudad de México,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
0,2022_Q1,3.953469,2.134665,2.992033,3.068,4.866799,2.872954,2.522971,2.479422,6.063560,...,3.136971,3.405845,2.939622,3.088067,5.042098,3.880562,4.292356,2.924682,2.139130,2.627598
1,2022_Q2,3.798665,2.427478,2.785745,2.953,4.131998,2.661397,2.709404,2.380596,5.266869,...,2.900986,3.248934,3.018120,3.119569,5.600978,3.416105,4.032327,2.954154,1.785905,3.140979
2,2022_Q3,3.697704,2.304535,2.555263,2.298,4.051126,2.462578,2.189561,2.312669,5.057697,...,2.807988,2.975509,2.908285,3.398284,4.985484,3.468522,3.509932,2.856816,1.894844,2.823368
3,2023_Q1,3.460535,1.936176,2.848725,1.762,3.554704,2.217715,2.485124,2.720658,3.900855,...,2.679430,2.339895,2.266381,3.122648,4.106569,2.853784,3.192391,2.756010,1.952460,3.211990
4,2023_Q2,3.183940,2.215882,2.740515,1.759,3.814756,2.287556,1.951570,2.395119,4.255903,...,2.413697,2.766792,2.277819,2.661481,3.938371,3.117355,3.580712,2.130368,1.687909,2.786777
5,2023_Q3,2.890444,2.188865,2.849253,1.819,3.674084,2.410967,2.426217,2.619587,3.930840,...,2.623781,2.724866,2.618993,2.590500,3.722493,3.170172,3.374651,1.848899,1.903655,2.470896
6,2024_Q1,3.489829,2.411825,2.282422,1.563,4.139268,2.183180,1.682406,2.234225,4.171803,...,2.327928,2.709234,2.083728,2.846898,3.695811,3.193434,3.123080,1.887316,1.690301,2.536135
7,2024_Q2,3.298298,2.635777,2.447040,1.998,3.822611,2.188133,1.697825,2.436339,3.907949,...,2.642396,2.589048,2.471620,3.779364,3.892234,3.343038,2.581364,1.990409,1.845911,2.759587
8,2024_Q3,3.839646,2.451764,2.254498,1.951,3.892254,2.196547,1.937326,2.431977,3.855282,...,2.460176,3.294232,2.410451,3.113554,4.181005,3.228869,3.271814,2.269870,1.860856,2.675315
9,2025_Q1,2.504293,2.460539,2.900015,2.066,3.570623,2.509461,2.445290,2.072312,3.631187,...,2.557826,3.746167,2.995943,2.794924,5.197182,3.517889,2.527558,2.228070,1.630410,2.674819


Now, using the .melt method we can reshape the DataFrame for it to only have three columns and two key id variables.

Regarding the .melt mehtod, this function is useful to reshape a DataFrame into a format where one or more columns are identifier variables (id_vars), while all other columns are considered measured variables (value_vars), and are “unpivoted” to the row axis, leaving just two non-identifier columns, ‘variable’ and ‘value’. The function has the following arguments

- id vars: The column name that won't be changed
- var_name: The name of the column associated to the original column names
- value_name: The name of the last column, associated to the values of the associated id vars and var_name

In [ ]:
df_inegi_desocupacion_v6 = df_inegi_desocupacion_v5.copy()
df_inegi_desocupacion_v6 = df_inegi_desocupacion_v6.melt(id_vars="Periodo", var_name="Estado", value_name="Desocupacion")    # If we ommit the last two arguments, the name of the two last columns is automatically generated
display(df_inegi_desocupacion_v6)

,Periodo,Estado,Desocupacion
0,2022_Q1,Aguascalientes,3.953469
1,2022_Q2,Aguascalientes,3.798665
2,2022_Q3,Aguascalientes,3.697704
3,2023_Q1,Aguascalientes,3.460535
4,2023_Q2,Aguascalientes,3.183940
...,...,...,...
379,2024_Q2,Zacatecas,2.759587
380,2024_Q3,Zacatecas,2.675315
381,2025_Q1,Zacatecas,2.674819
382,2025_Q2,Zacatecas,1.886838


### Definition of pivot and pivot tables - TAREA 

A **pivot table** is a table of values which are aggregations of group of individual values from a more extensive table within one or more discrete categories.

#### But what do we mean by this?

A pivot table essentially reshapes the data of a table (usualy from a longer format into a wider, more readable one); while a normal table has normally one key ID, a pivot table has two or more. The extra key ID's are created by agreggating the values of discrete categories. That is, it lets us form sets of sets in a tree-like way, so we can display the aggregated values of each of its branches.

#### And to what do we refer by a pivot?

While pivot tables refer to the specific tables resulting from re-shaping a database (or other table), a **pivot** referts to the action of transforming/re-shaping the data.

The singular version of our DataFrame just obtained is useful to develop pivot tables. This we do next:

In [82]:
df_inegi_desocupacion_v7 = df_inegi_desocupacion_v6.copy()
df_inegi_desocupacion_v7 = df_inegi_desocupacion_v7.pivot(index="Estado", columns="Periodo", values = "Desocupacion").sort_index().reset_index()
df_inegi_desocupacion_v7.index.name = None
df_inegi_desocupacion_v7.columns.name = None
display(df_inegi_desocupacion_v7)

,Estado,2022_Q1,2022_Q2,2022_Q3,2023_Q1,2023_Q2,2023_Q3,2024_Q1,2024_Q2,2024_Q3,2025_Q1,2025_Q2,2025_Q3
0,Aguascalientes,3.953469,3.798665,3.697704,3.460535,3.183940,2.890444,3.489829,3.298298,3.839646,2.504293,2.946359,2.859346
1,Baja California,2.134665,2.427478,2.304535,1.936176,2.215882,2.188865,2.411825,2.635777,2.451764,2.460539,2.226564,2.434564
2,Baja California Sur,2.992033,2.785745,2.555263,2.848725,2.740515,2.849253,2.282422,2.447040,2.254498,2.900015,2.391026,2.390711
3,Campeche,3.068000,2.953000,2.298000,1.762000,1.759000,1.819000,1.563000,1.998000,1.951000,2.066000,2.612000,2.903000
4,Chiapas,2.522971,2.709404,2.189561,2.485124,1.951570,2.426217,1.682406,1.697825,1.937326,2.445290,2.464905,2.440461
5,Chihuahua,2.479422,2.380596,2.312669,2.720658,2.395119,2.619587,2.234225,2.436339,2.431977,2.072312,2.228869,2.130321
6,Ciudad de México,6.063560,5.266869,5.057697,3.900855,4.255903,3.930840,4.171803,3.907949,3.855282,3.631187,3.816314,3.782206
7,Coahuila de Zaragoza,4.866799,4.131998,4.051126,3.554704,3.814756,3.674084,4.139268,3.822611,3.892254,3.570623,3.830592,3.706578
8,Colima,2.872954,2.661397,2.462578,2.217715,2.287556,2.410967,2.183180,2.188133,2.196547,2.509461,1.827467,2.351820
9,Durango,3.481090,3.278500,3.151711,2.897101,2.930798,2.990042,2.945057,2.906285,3.169745,3.320484,2.771744,2.832379


We'll need the data from the csv file from the previous class to make comparisons. Here we prep said data:

In [88]:
df_pob.columns

Index(['ENT', 'NOM_ENT ', 'POBTOT', 'POBFEM', 'POBMAS', 'P_0A2', 'P_0A2_F',
       'P_0A2_M', 'P_0A17', 'P_3YMAS',
       ...
       'VPH_TELEF', 'VPH_CEL', 'VPH_INTER', 'VPH_STVP', 'VPH_SPMVPI',
       'VPH_CVJ', 'VPH_SINRTV', 'VPH_SINLTC', 'VPH_SINCIN', 'VPH_SINTIC'],
      dtype='object', length=222)

In [95]:
df_pob_v2 = df_pob
df_pob_v2.columns = df_pob_v2.columns.str.strip()
df_pob_v2 = df_pob_v2.loc[:, ["ENT", "NOM_ENT", "POBTOT", "P_15YMAS"]]
display(df_pob_v2)


,ENT,NOM_ENT,POBTOT,P_15YMAS
0,1,Aguascalientes,1425607,1038904
1,2,Baja California,3769020,2882498
2,3,Baja California Sur,798447,597552
3,4,Campeche,928363,682951
4,5,Coahuila de Zaragoza,3146771,2316332
5,6,Colima,731391,556272
6,7,Chiapas,5543828,3745908
7,8,Chihuahua,3741869,2791907
8,9,Ciudad de México,9209944,7547545
9,10,Durango,1832650,1315571


## Types of Joins - Tarea

---

Pandas can hadle 4 types of join operations on DataFrames: Inner, left, right, full outer and index join. These are akin to common set operations, and can be illustrated with Venn Diagrams. 

### Inner Join

Pandas inner join returns a DataFrame with only those rows that have common charactersitics. The general form of the join command is `df = pd.merge(df_01, df_02, on='id', how='inner')`

This is simmilar to the intersection of sets. For reference see the attached immage:

![image](../ATTACHMENTS/imagen_02_innerJoin_Clase_03.jpg)

### Left Join

Pandas left join returns a DataFrame where all the records of the first DataFrame will be displayed, but for the second DataFrame only the records that share a key with the first will be displayed. The general form of the join command is `df = pd.merge(df_01, df_02, on='id', how='left')`

For reference see the attached image:

![image](../ATTACHMENTS/imagen_03_LeftJoin_Clase_03.jpg)

### Right join

Pandas right join behaves like the left join, but interchanging the first and the second DataFrame. The general form of the join command is `df = pd.merge(df_01, df_02, on='id', how='right')`

For reference see the attached image:

![image](../ATTACHMENTS/imagen_04_RightJoin_Clase_03.jpg)

### Full outer join

Pandas full outer join returns all the rows from the left and right DataFrame, matching rows whenever possible, placing NaN otherwise. The general form of the join command is `df = pd.merge(df_01, df_02, on='id', how='outer')`

For reference see the attached image:

![image](../ATTACHMENTS/imagen_05_FullOuterJoin_Clase_03.jpg)

### Index Join

Pandas index joins merges DataFrames on indices. When using this join both the Dataframes are merged on an index using default Inner Join. The general from of the join command is `pd.merge(df_01, df_02, left_index=True, right_index=True)`

We now used this newly gained knowledge to join both DataFrames:

In [107]:
df_join_pob_desocup = df_pob_v2.merge(df_inegi_desocupacion_v7, how="left", left_on="NOM_ENT",  right_on="Estado")    # In my left DataFrame on the column "NOM_ENT" search for matches with the column "ESTADO" 
                                                                                                                      # on the right DataFrame
df_join_pob_desocup = df_join_pob_desocup.drop(columns = "Estado")
df_join_pob_desocup = df_join_pob_desocup.rename(columns={"P_15YMAS" : "PEA"})
display(df_join_pob_desocup)                                                                                                                    

,ENT,NOM_ENT,POBTOT,PEA,2022_Q1,2022_Q2,2022_Q3,2023_Q1,2023_Q2,2023_Q3,2024_Q1,2024_Q2,2024_Q3,2025_Q1,2025_Q2,2025_Q3
0,1,Aguascalientes,1425607,1038904,3.953469,3.798665,3.697704,3.460535,3.183940,2.890444,3.489829,3.298298,3.839646,2.504293,2.946359,2.859346
1,2,Baja California,3769020,2882498,2.134665,2.427478,2.304535,1.936176,2.215882,2.188865,2.411825,2.635777,2.451764,2.460539,2.226564,2.434564
2,3,Baja California Sur,798447,597552,2.992033,2.785745,2.555263,2.848725,2.740515,2.849253,2.282422,2.447040,2.254498,2.900015,2.391026,2.390711
3,4,Campeche,928363,682951,3.068000,2.953000,2.298000,1.762000,1.759000,1.819000,1.563000,1.998000,1.951000,2.066000,2.612000,2.903000
4,5,Coahuila de Zaragoza,3146771,2316332,4.866799,4.131998,4.051126,3.554704,3.814756,3.674084,4.139268,3.822611,3.892254,3.570623,3.830592,3.706578
5,6,Colima,731391,556272,2.872954,2.661397,2.462578,2.217715,2.287556,2.410967,2.183180,2.188133,2.196547,2.509461,1.827467,2.351820
6,7,Chiapas,5543828,3745908,2.522971,2.709404,2.189561,2.485124,1.951570,2.426217,1.682406,1.697825,1.937326,2.445290,2.464905,2.440461
7,8,Chihuahua,3741869,2791907,2.479422,2.380596,2.312669,2.720658,2.395119,2.619587,2.234225,2.436339,2.431977,2.072312,2.228869,2.130321
8,9,Ciudad de México,9209944,7547545,6.063560,5.266869,5.057697,3.900855,4.255903,3.930840,4.171803,3.907949,3.855282,3.631187,3.816314,3.782206
9,10,Durango,1832650,1315571,3.481090,3.278500,3.151711,2.897101,2.930798,2.990042,2.945057,2.906285,3.169745,3.320484,2.771744,2.832379


## Analysis

We've finally reached the point were we can perform analysis on the transformed data.

Our first proposed analysis is to clasify all states as small, median, big, and very big, according to its total population. To define the threshold for each label, we will use the intervals between the quartiles. To do this we first need to compute said quartiles. This is done next:

In [108]:
df_join_pob_desocup["POBTOT"] = pd.to_numeric(df_join_pob_desocup["POBTOT"], errors="coerce")
q = df_join_pob_desocup["POBTOT"].quantile([0.25, 0.5, 0.75])
q1 = q.loc[0.25]
q2 = q.loc[0.50]
q3 = q.loc[0.75]

We now use the pandas function .cut place the labels in a new column "tamano_pob", depending on which interval does the associated total population value lies (where the intervals are determined by the calculated quartiles).

In [110]:
df_join_pob_desocup["tamano_pob"] = pd.cut(df_join_pob_desocup["POBTOT"], bins = [-np.inf, q1, q2, q3, np.inf], labels = ["pequeno", "mediano", "grande", "muy_grande"], include_lowest=True)
display(df_join_pob_desocup[["NOM_ENT", "POBTOT", "tamano_pob"]])

,NOM_ENT,POBTOT,tamano_pob
0,Aguascalientes,1425607,pequeno
1,Baja California,3769020,grande
2,Baja California Sur,798447,pequeno
3,Campeche,928363,pequeno
4,Coahuila de Zaragoza,3146771,grande
5,Colima,731391,pequeno
6,Chiapas,5543828,muy_grande
7,Chihuahua,3741869,grande
8,Ciudad de México,9209944,muy_grande
9,Durango,1832650,pequeno


Ahora, para ver el minimo, la media, la mediana y el maximo de desocupacion de cada estado usamos la funcion  groupby y el metodo .agg. De esta manera definimos un nuevo DataFrame con solo 4 filas (las etiquetas) y 5 columnas (las etiquetas y los cuartro estadisticos descriptivos).

In [120]:
df_join_pob_desocup_gb = df_join_pob_desocup.groupby("tamano_pob")["2025_Q3"].agg(minimo_2025_Q3 = "min", media_2025_Q3 = "mean", mediana_2025_Q3 = "median", maximo_2025_Q3 = "max").reset_index()
display(df_join_pob_desocup_gb)

C:\Users\ossia\AppData\Local\Temp\ipykernel_23028\1331430669.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_join_pob_desocup_gb = df_join_pob_desocup.groupby("tamano_pob")["2025_Q3"].agg(minimo_2025_Q3 = "min", media_2025_Q3 = "mean", mediana_2025_Q3 = "median", maximo_2025_Q3 = "max").reset_index()


,tamano_pob,minimo_2025_Q3,media_2025_Q3,mediana_2025_Q3,maximo_2025_Q3
0,pequeno,2.240728,2.647341,2.701793,3.029533
1,mediano,1.382300,2.578230,2.540167,4.601016
2,grande,1.171000,2.256739,2.182396,3.706578
3,muy_grande,2.058673,2.732437,2.634931,3.782206


### Ejercicio Tarea

- Parte 1: Mediante groupby y join anade las columnas correspondientes al periodo anterior y el periodo de un ano atras (mismo ejercicio y concatenar los DataFrames resultantes)
- Parte 2: Replicar la parte 1 pero ahora obteniendo la cantidad de poblacion inactiva, no el porcentaje
- Parte 3: De la parte 2, obtener la variacion de poblacion desocupada respecto al periodo de un ano atras

- Extra: Repasar celda por celda y anadir un 'Markdown' antes de cada celda, que explique que transformacion ocurrio y cuales fueron las funciones clave que se utilizaron, y para que se utilizaron.

Obtener conclusiones

### Ejercicio Tarea - Parte 1

Usamos groupby y .agg para obtener los DataFrames afines los dos periodos anteriores. El procedimiento es el mismo al usado para el 3er quatrimetre del 2025.

In [121]:
df_join_pob_desocup_gb_02 = df_join_pob_desocup.groupby("tamano_pob")["2025_Q2"].agg(minimo_2025_Q2 = "min", media_2025_Q2 = "mean", mediana_2025_Q2 = "median", maximo_2025_Q2 = "max").reset_index()
display(df_join_pob_desocup_gb_02)

C:\Users\ossia\AppData\Local\Temp\ipykernel_23028\1046567420.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_join_pob_desocup_gb_02 = df_join_pob_desocup.groupby("tamano_pob")["2025_Q2"].agg(minimo_2025_Q2 = "min", media_2025_Q2 = "mean", mediana_2025_Q2 = "median", maximo_2025_Q2 = "max").reset_index()


,tamano_pob,minimo_2025_Q2,media_2025_Q2,mediana_2025_Q2,maximo_2025_Q2
0,pequeno,1.827467,2.426557,2.488512,2.946359
1,mediano,1.698092,2.663401,2.459238,4.248820
2,grande,0.967000,2.164214,1.980560,3.830592
3,muy_grande,2.272482,2.832376,2.751339,3.816314


In [122]:
df_join_pob_desocup_gb_03 = df_join_pob_desocup.groupby("tamano_pob")["2025_Q1"].agg(minimo_2025_Q1 = "min", media_2025_Q1 = "mean", mediana_2025_Q1 = "median", maximo_2025_Q1 = "max").reset_index()
display(df_join_pob_desocup_gb_03)

C:\Users\ossia\AppData\Local\Temp\ipykernel_23028\119246801.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_join_pob_desocup_gb_03 = df_join_pob_desocup.groupby("tamano_pob")["2025_Q1"].agg(minimo_2025_Q1 = "min", media_2025_Q1 = "mean", mediana_2025_Q1 = "median", maximo_2025_Q1 = "max").reset_index()


,tamano_pob,minimo_2025_Q1,media_2025_Q1,mediana_2025_Q1,maximo_2025_Q1
0,pequeno,2.066000,2.638427,2.566171,3.320484
1,mediano,1.271448,2.791737,2.676375,5.197182
2,grande,0.883000,2.176894,2.012698,3.570623
3,muy_grande,1.987277,2.677585,2.723409,3.631187


Ahora concatenamos usando la fuincion merge. Hacemos un joint tipo outer (en este caso no importa, pues con cualquier join obtendremos lo mismo).

In [125]:
df_join_pob_desocup_gb_join = df_join_pob_desocup_gb_03.merge(df_join_pob_desocup_gb_02, how = "outer", left_on="tamano_pob", right_on="tamano_pob").merge(df_join_pob_desocup_gb, how = "outer", left_on="tamano_pob", right_on="tamano_pob")
display(df_join_pob_desocup_gb_join)

,tamano_pob,minimo_2025_Q1,media_2025_Q1,mediana_2025_Q1,maximo_2025_Q1,minimo_2025_Q2,media_2025_Q2,mediana_2025_Q2,maximo_2025_Q2,minimo_2025_Q3,media_2025_Q3,mediana_2025_Q3,maximo_2025_Q3
0,pequeno,2.066000,2.638427,2.566171,3.320484,1.827467,2.426557,2.488512,2.946359,2.240728,2.647341,2.701793,3.029533
1,mediano,1.271448,2.791737,2.676375,5.197182,1.698092,2.663401,2.459238,4.248820,1.382300,2.578230,2.540167,4.601016
2,grande,0.883000,2.176894,2.012698,3.570623,0.967000,2.164214,1.980560,3.830592,1.171000,2.256739,2.182396,3.706578
3,muy_grande,1.987277,2.677585,2.723409,3.631187,2.272482,2.832376,2.751339,3.816314,2.058673,2.732437,2.634931,3.782206


Analizando la tabla resultante podemos observar que la tasa de desocupacion no ha variado en gran medida a lo largo de los 3 cuatrimestres analizados del 2025. Si acaso podemos mencionar que la tasa de desocupacion maxima de los estados con un tamano de poblacion mediano ha aumentado significativamente. Esto indica que hay algun estado dentro de la lista de estados medianos que puede estar teniendo una recesion en su mercado laboral. Esto podria motivar un estudio mas exhaustivo.

### Ejercicio Tarea - Parte 2

We make a new copy of the complete pivoted DataFrame

In [145]:
df_join_pob_desocup_tot = df_join_pob_desocup.copy()
display(df_join_pob_desocup_tot)

,ENT,NOM_ENT,POBTOT,PEA,2022_Q1,2022_Q2,2022_Q3,2023_Q1,2023_Q2,2023_Q3,2024_Q1,2024_Q2,2024_Q3,2025_Q1,2025_Q2,2025_Q3,tamano_pob
0,1,Aguascalientes,1425607,1038904,3.953469,3.798665,3.697704,3.460535,3.183940,2.890444,3.489829,3.298298,3.839646,2.504293,2.946359,2.859346,pequeno
1,2,Baja California,3769020,2882498,2.134665,2.427478,2.304535,1.936176,2.215882,2.188865,2.411825,2.635777,2.451764,2.460539,2.226564,2.434564,grande
2,3,Baja California Sur,798447,597552,2.992033,2.785745,2.555263,2.848725,2.740515,2.849253,2.282422,2.447040,2.254498,2.900015,2.391026,2.390711,pequeno
3,4,Campeche,928363,682951,3.068000,2.953000,2.298000,1.762000,1.759000,1.819000,1.563000,1.998000,1.951000,2.066000,2.612000,2.903000,pequeno
4,5,Coahuila de Zaragoza,3146771,2316332,4.866799,4.131998,4.051126,3.554704,3.814756,3.674084,4.139268,3.822611,3.892254,3.570623,3.830592,3.706578,grande
5,6,Colima,731391,556272,2.872954,2.661397,2.462578,2.217715,2.287556,2.410967,2.183180,2.188133,2.196547,2.509461,1.827467,2.351820,pequeno
6,7,Chiapas,5543828,3745908,2.522971,2.709404,2.189561,2.485124,1.951570,2.426217,1.682406,1.697825,1.937326,2.445290,2.464905,2.440461,muy_grande
7,8,Chihuahua,3741869,2791907,2.479422,2.380596,2.312669,2.720658,2.395119,2.619587,2.234225,2.436339,2.431977,2.072312,2.228869,2.130321,grande
8,9,Ciudad de México,9209944,7547545,6.063560,5.266869,5.057697,3.900855,4.255903,3.930840,4.171803,3.907949,3.855282,3.631187,3.816314,3.782206,muy_grande
9,10,Durango,1832650,1315571,3.481090,3.278500,3.151711,2.897101,2.930798,2.990042,2.945057,2.906285,3.169745,3.320484,2.771744,2.832379,pequeno


Ahora, para obtener el total de poblacion desocupada, multiplicamos los porcentajes asociados con la poblacion mayor de 15 anos de cada estado. Hacemos esto para los 3 quatrimestres del 2025.

In [146]:
df_join_pob_desocup_tot["2025_Q1"] = df_join_pob_desocup["PEA"] * df_join_pob_desocup["2025_Q1"]
df_join_pob_desocup_tot["2025_Q2"] = df_join_pob_desocup["PEA"] * df_join_pob_desocup["2025_Q2"]
df_join_pob_desocup_tot["2025_Q3"] = df_join_pob_desocup["PEA"] * df_join_pob_desocup["2025_Q3"]
display(df_join_pob_desocup_tot)

,ENT,NOM_ENT,POBTOT,PEA,2022_Q1,2022_Q2,2022_Q3,2023_Q1,2023_Q2,2023_Q3,2024_Q1,2024_Q2,2024_Q3,2025_Q1,2025_Q2,2025_Q3,tamano_pob
0,1,Aguascalientes,1425607,1038904,3.953469,3.798665,3.697704,3.460535,3.183940,2.890444,3.489829,3.298298,3.839646,2.601720e+06,3.060984e+06,2.970586e+06,pequeno
1,2,Baja California,3769020,2882498,2.134665,2.427478,2.304535,1.936176,2.215882,2.188865,2.411825,2.635777,2.451764,7.092500e+06,6.418065e+06,7.017626e+06,grande
2,3,Baja California Sur,798447,597552,2.992033,2.785745,2.555263,2.848725,2.740515,2.849253,2.282422,2.447040,2.254498,1.732910e+06,1.428762e+06,1.428574e+06,pequeno
3,4,Campeche,928363,682951,3.068000,2.953000,2.298000,1.762000,1.759000,1.819000,1.563000,1.998000,1.951000,1.410977e+06,1.783868e+06,1.982607e+06,pequeno
4,5,Coahuila de Zaragoza,3146771,2316332,4.866799,4.131998,4.051126,3.554704,3.814756,3.674084,4.139268,3.822611,3.892254,8.270748e+06,8.872922e+06,8.585666e+06,grande
5,6,Colima,731391,556272,2.872954,2.661397,2.462578,2.217715,2.287556,2.410967,2.183180,2.188133,2.196547,1.395943e+06,1.016569e+06,1.308252e+06,pequeno
6,7,Chiapas,5543828,3745908,2.522971,2.709404,2.189561,2.485124,1.951570,2.426217,1.682406,1.697825,1.937326,9.159831e+06,9.233307e+06,9.141742e+06,muy_grande
7,8,Chihuahua,3741869,2791907,2.479422,2.380596,2.312669,2.720658,2.395119,2.619587,2.234225,2.436339,2.431977,5.785703e+06,6.222795e+06,5.947658e+06,grande
8,9,Ciudad de México,9209944,7547545,6.063560,5.266869,5.057697,3.900855,4.255903,3.930840,4.171803,3.907949,3.855282,2.740655e+07,2.880381e+07,2.854637e+07,muy_grande
9,10,Durango,1832650,1315571,3.481090,3.278500,3.151711,2.897101,2.930798,2.990042,2.945057,2.906285,3.169745,4.368332e+06,3.646427e+06,3.726196e+06,pequeno


Una vez que tenemos el DataFrame con los datos adecuados, procedemos a usar la funcion groupby y el metodo .agg para obtener cada una de los DataFrames con los estadisticos basicos:

In [154]:
df_join_pob_desocup_tot_gp_01 = df_join_pob_desocup_tot.groupby("tamano_pob")["2025_Q1"].agg( minimo_pob_desocup_2025_Q1 = "min", media_pob_desocup_2025_Q1 = "mean", mediana_pob_desocup_2025_Q1 = "median", maximo_pob_desocup_2025_Q1 = "max").reset_index()
display(df_join_pob_desocup_tot_gp_01)

C:\Users\ossia\AppData\Local\Temp\ipykernel_23028\1974387150.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_join_pob_desocup_tot_gp_01 = df_join_pob_desocup_tot.groupby("tamano_pob")["2025_Q1"].agg( minimo_pob_desocup_2025_Q1 = "min", media_pob_desocup_2025_Q1 = "mean", mediana_pob_desocup_2025_Q1 = "median", maximo_pob_desocup_2025_Q1 = "max").reset_index()


,tamano_pob,minimo_pob_desocup_2025_Q1,media_pob_desocup_2025_Q1,mediana_pob_desocup_2025_Q1,maximo_pob_desocup_2025_Q1
0,pequeno,1.395943e+06,2.428999e+06,2.407686e+06,4.368332e+06
1,mediano,1.909741e+06,5.259498e+06,5.002592e+06,9.085137e+06
2,grande,2.199824e+06,5.820827e+06,5.326541e+06,9.324320e+06
3,muy_grande,9.159831e+06,1.713710e+07,1.290010e+07,3.690436e+07


In [155]:
df_join_pob_desocup_tot_gp_02 = df_join_pob_desocup_tot.groupby("tamano_pob")["2025_Q2"].agg( minimo_pob_desocup_2025_Q2 = "min", media_pob_desocup_2025_Q2 = "mean", mediana_pob_desocup_2025_Q2 = "median", maximo_pob_desocup_2025_Q2 = "max").reset_index()
display(df_join_pob_desocup_tot_gp_02)

C:\Users\ossia\AppData\Local\Temp\ipykernel_23028\3899398758.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_join_pob_desocup_tot_gp_02 = df_join_pob_desocup_tot.groupby("tamano_pob")["2025_Q2"].agg( minimo_pob_desocup_2025_Q2 = "min", media_pob_desocup_2025_Q2 = "mean", mediana_pob_desocup_2025_Q2 = "median", maximo_pob_desocup_2025_Q2 = "max").reset_index()


,tamano_pob,minimo_pob_desocup_2025_Q2,media_pob_desocup_2025_Q2,mediana_pob_desocup_2025_Q2,maximo_pob_desocup_2025_Q2
0,pequeno,1.016569e+06,2.224894e+06,2.227933e+06,3.646427e+06
1,mediano,2.550570e+06,4.952112e+06,4.519561e+06,7.676460e+06
2,grande,2.409094e+06,5.790645e+06,5.882432e+06,9.368441e+06
3,muy_grande,9.233307e+06,1.825924e+07,1.370969e+07,3.946962e+07


In [156]:
df_join_pob_desocup_tot_gp_03 = df_join_pob_desocup_tot.groupby("tamano_pob")["2025_Q3"].agg( minimo_pob_desocup_2025_Q3 = "min", media_pob_desocup_2025_Q3 = "mean", mediana_pob_desocup_2025_Q3 = "median", maximo_pob_desocup_2025_Q3 = "max").reset_index()
display(df_join_pob_desocup_tot_gp_03)

C:\Users\ossia\AppData\Local\Temp\ipykernel_23028\3587066608.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_join_pob_desocup_tot_gp_03 = df_join_pob_desocup_tot.groupby("tamano_pob")["2025_Q3"].agg( minimo_pob_desocup_2025_Q3 = "min", media_pob_desocup_2025_Q3 = "mean", mediana_pob_desocup_2025_Q3 = "median", maximo_pob_desocup_2025_Q3 = "max").reset_index()


,tamano_pob,minimo_pob_desocup_2025_Q3,media_pob_desocup_2025_Q3,mediana_pob_desocup_2025_Q3,maximo_pob_desocup_2025_Q3
0,pequeno,1.308252e+06,2.432993e+06,2.264501e+06,3.726196e+06
1,mediano,2.141647e+06,4.787357e+06,4.648136e+06,8.042985e+06
2,grande,2.917320e+06,6.043356e+06,5.931128e+06,8.585666e+06
3,muy_grande,9.141742e+06,1.783527e+07,1.280952e+07,4.188547e+07


Finalmente, usamos un outer join usando la funcion merge para juntar todo en un mismo DataFrame:

In [157]:
df_join_pob_desocup_tot_gp_join = df_join_pob_desocup_tot_gp_01.merge(df_join_pob_desocup_tot_gp_02, how = "outer", left_on="tamano_pob", right_on="tamano_pob").merge(df_join_pob_desocup_tot_gp_03, how = "outer", left_on="tamano_pob", right_on="tamano_pob")
display(df_join_pob_desocup_tot_gp_join)

,tamano_pob,minimo_pob_desocup_2025_Q1,media_pob_desocup_2025_Q1,mediana_pob_desocup_2025_Q1,maximo_pob_desocup_2025_Q1,minimo_pob_desocup_2025_Q2,media_pob_desocup_2025_Q2,mediana_pob_desocup_2025_Q2,maximo_pob_desocup_2025_Q2,minimo_pob_desocup_2025_Q3,media_pob_desocup_2025_Q3,mediana_pob_desocup_2025_Q3,maximo_pob_desocup_2025_Q3
0,pequeno,1.395943e+06,2.428999e+06,2.407686e+06,4.368332e+06,1.016569e+06,2.224894e+06,2.227933e+06,3.646427e+06,1.308252e+06,2.432993e+06,2.264501e+06,3.726196e+06
1,mediano,1.909741e+06,5.259498e+06,5.002592e+06,9.085137e+06,2.550570e+06,4.952112e+06,4.519561e+06,7.676460e+06,2.141647e+06,4.787357e+06,4.648136e+06,8.042985e+06
2,grande,2.199824e+06,5.820827e+06,5.326541e+06,9.324320e+06,2.409094e+06,5.790645e+06,5.882432e+06,9.368441e+06,2.917320e+06,6.043356e+06,5.931128e+06,8.585666e+06
3,muy_grande,9.159831e+06,1.713710e+07,1.290010e+07,3.690436e+07,9.233307e+06,1.825924e+07,1.370969e+07,3.946962e+07,9.141742e+06,1.783527e+07,1.280952e+07,4.188547e+07


Analizando las cifras totales en vez de las porcentuales nos permite tener una perspectiva mas clara de las dinamicas de desocupacion en las diferentes categorias. Una observacion que de otra manera habria sido complicada de identificar es, por ejemplo, el hecho de que la media y la mediana de poblacion desocupada entre estados de tamano poblacional mediano y grande es casi la misma.

### Ejercicio Tarea - Parte 3 - PENDIENTE